# Cabaran: Menganalisis Teks tentang Sains Data

Dalam contoh ini, mari lakukan satu latihan mudah yang merangkumi semua langkah proses sains data tradisional. Anda tidak perlu menulis sebarang kod, anda hanya boleh klik pada sel di bawah untuk melaksanakannya dan memerhati hasilnya. Sebagai cabaran, anda digalakkan untuk mencuba kod ini dengan data yang berbeza.

## Matlamat

Dalam pelajaran ini, kita telah membincangkan pelbagai konsep berkaitan Sains Data. Mari cuba untuk menemui lebih banyak konsep berkaitan dengan melakukan **perlombongan teks**. Kita akan bermula dengan sebuah teks tentang Sains Data, mengekstrak kata kunci daripadanya, dan kemudian cuba memvisualisasikan keputusan.

Sebagai teks, saya akan menggunakan halaman mengenai Sains Data dari Wikipedia:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## Langkah 1: Mendapatkan Data

Langkah pertama dalam setiap proses sains data adalah mendapatkan data. Kami akan menggunakan perpustakaan `requests` untuk melakukan itu:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## Langkah 2: Menukar Data

Langkah seterusnya adalah untuk menukar data ke dalam bentuk yang sesuai untuk diproses. Dalam kes kami, kami telah memuat turun kod sumber HTML dari halaman, dan kami perlu menukarnya ke dalam teks biasa.

Terdapat banyak cara untuk melakukan ini. Kami akan menggunakan [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/), sebuah perpustakaan Python yang popular untuk memparsing HTML. BeautifulSoup membolehkan kami menarget elemen HTML tertentu, jadi kami boleh fokus pada kandungan artikel utama dari Wikipedia dan mengurangkan beberapa menu navigasi, bar sisi, kaki halaman, dan kandungan yang tidak relevan lain (walaupun beberapa teks templat mungkin masih kekal).


Pertama, kita perlu memasang perpustakaan BeautifulSoup untuk penguraian HTML:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## Langkah 3: Mendapatkan Wawasan

Langkah paling penting adalah menukar data kita ke dalam suatu bentuk dari mana kita boleh mengambil wawasan. Dalam kes kita, kita mahu mengekstrak kata kunci dari teks, dan melihat kata kunci mana yang lebih bermakna.

Kita akan menggunakan perpustakaan Python yang dipanggil [RAKE](https://github.com/aneesha/RAKE) untuk pengekstrakan kata kunci. Pertama, mari pasang perpustakaan ini sekiranya ia tidak ada: 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

Fungsi utama tersedia daripada objek `Rake`, yang boleh kita sesuaikan menggunakan beberapa parameter. Dalam kes kami, kami akan menetapkan panjang minimum kata kunci kepada 5 aksara, kekerapan minimum kata kunci dalam dokumen kepada 3, dan bilangan maksimum perkataan dalam kata kunci - kepada 2. Sila cuba bermain dengan nilai lain dan perhatikan hasilnya.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


Kami memperoleh senarai terma bersama-sama dengan darjah kepentingan yang berkaitan. Seperti yang anda lihat, disiplin yang paling relevan, seperti pembelajaran mesin dan data besar, ada dalam senarai di kedudukan teratas.

## Langkah 4: Memvisualisasikan Hasil

Orang mampu mentafsir data dengan lebih baik dalam bentuk visual. Oleh itu, sering masuk akal untuk memvisualisasikan data bagi mendapatkan beberapa pandangan. Kita boleh menggunakan perpustakaan `matplotlib` dalam Python untuk melukis pengagihan mudah kata kunci dengan kepentingannya:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

Walau bagaimanapun, terdapat cara yang lebih baik untuk memvisualisasikan kekerapan perkataan - menggunakan **Awan Perkataan**. Kita akan perlu memasang satu perpustakaan lain untuk memplot awan perkataan dari senarai kata kunci kita.


In [ ]:
!{sys.executable} -m pip install wordcloud

Objek `WordCloud` bertanggungjawab untuk menerima sama ada teks asal, atau senarai perkataan yang telah dikira kekerapan, dan mengembalikan imej, yang kemudian boleh dipaparkan menggunakan `matplotlib`:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

Kami juga boleh memasukkan teks asal ke dalam `WordCloud` - mari kita lihat jika kita dapat menghasilkan keputusan yang serupa:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

Anda boleh lihat bahawa awan perkataan kini kelihatan lebih mengagumkan, tetapi ia juga mengandungi banyak bunyi gangguan (contohnya, perkataan yang tidak berkaitan seperti `Retrieved on`). Selain itu, kami mendapat lebih sedikit kata kunci yang terdiri daripada dua perkataan, seperti *data scientist*, atau *computer science*. Ini kerana algoritma RAKE melakukan tugas yang lebih baik dalam memilih kata kunci yang baik daripada teks. Contoh ini menggambarkan pentingnya penyediaan dan pembersihan data, kerana gambaran yang jelas pada akhirnya akan membolehkan kita membuat keputusan yang lebih baik.

Dalam latihan ini kami telah melalui proses mudah mengekstrak makna daripada teks Wikipedia, dalam bentuk kata kunci dan awan perkataan. Contoh ini agak mudah, tetapi ia menunjukkan dengan baik semua langkah tipikal yang akan diambil oleh seorang saintis data apabila bekerja dengan data, bermula dari pemerolehan data, sehingga visualisasi.

Dalam kursus kami, kami akan membincangkan semua langkah tersebut secara terperinci.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila ambil maklum bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya harus dianggap sebagai sumber yang sahih. Untuk maklumat penting, terjemahan oleh manusia profesional adalah disyorkan. Kami tidak bertanggungjawab terhadap sebarang salah faham atau salah tafsir yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
